In [1]:
import pytest
from pytest_check import check 

import polars as pl
import ibis.expr.types as ir
from mountainash_utils_rules import apply_context_rules_engine_ibis  # Replace `your_module` with the actual module name
from dataclasses import dataclass
from typing import Optional

from mountainash_data import BaseDataFrame, DataFrameUtils, IbisDataFrame

UNKNOWN = "<NA>"

# import pytest
# from module_name import apply_context_rules_engine_ibis, rules, dimensions, context
@dataclass
class context:
    rule_name:  Optional[str]
    DIM_1:      Optional[str]
    DIM_2:      Optional[str]
    DIM_3:      Optional[str]

CONTEXT = context(rule_name="rule_1", DIM_1="A", DIM_2="2", DIM_3=UNKNOWN)


raw_rules = pl.DataFrame({  "rule_name": ["rule_1", "rule_2", "rule_3"],
                        "DIM_1": ["A", "B", "C"],
                        "DIM_2": ["1", "2", "3"],
                        "DIM_3": ["X", UNKNOWN, UNKNOWN]
                    })
df_rules = IbisDataFrame(raw_rules)

dimensions = [
            # "DIM_1", 
             "DIM_2", 
             "DIM_3"
            ]

context_value = getattr(CONTEXT, "DIM_1")
print(CONTEXT)

context(rule_name='rule_1', DIM_1='A', DIM_2='2', DIM_3='<NA>')


In [2]:
rules = apply_context_rules_engine_ibis(CONTEXT=CONTEXT, rules=df_rules, dimensions=dimensions, keep_all=True)
rules.execute()


A
Active dimensions: ['DIM_3', 'DIM_2']
Evaluating dimension: DIM_3
context_value: <NA>
Evaluating dimension: DIM_2
context_value: 2


,rule_name,DIM_1,DIM_2,DIM_3,rule_softmatch_count,context_softmatch_count,dual_softmatch_count,hard_match_count,dropped,dropped_by,filter_all_false,filter_all_true,filter1,filter2,filter3,filter_product,any_false,any_true,keep
0,rule_1,A,1,X,0,1,0,0,True,DIM_2,False,True,5,5,3,75,True,False,False
1,rule_2,B,2,<NA>,1,1,1,2,None,None,False,True,5,5,2,50,False,True,True
2,rule_3,C,3,<NA>,1,1,1,1,True,DIM_2,False,True,5,5,3,75,True,False,False


In [ ]:
dimension_tests = [
    ("DIM_1", 1),
    ("DIM_2", 1),
    ("DIM_3", 2),
    ("DIM_4", 0)
]
@pytest.mark.parametrize("dimension, count_matching", dimension_tests)
def test_dim1(dimension, count_matching):
        
    dimensions = [dimension]
    rules = apply_context_rules_engine_ibis(context, df_rules, dimensions, keep_all=False)

    print(rules)

    with check:
        assert rules.count() == count_matching




def test_apply_context_rules_engine_ibis_no_rules_specified():
    empty_rules = pl.DataFrame({})
    df_empty_rules = IbisDataFrame(empty_rules)

    with pytest.raises(ValueError):
        apply_context_rules_engine_ibis(context, df_empty_rules, dimensions)

def test_apply_context_rules_engine_ibis_result_type():
    result = apply_context_rules_engine_ibis(context, rules, dimensions)
    assert isinstance(result, BaseDataFrame)